<a href="https://colab.research.google.com/github/kamal-gavel/Applicability-of-Limit-Continuity-Discontinuity-Framework-on-Sentiment-Based-Stock-Direction-Predic/blob/main/%E2%80%9CA_Limit%E2%80%93Continuity%E2%80%93Discontinuity_Framework_for_Multi_Horizon_Quantitative_Trading_using_Machine_Learning%E2%80%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# 📌 FINAL ELITE QUANT SYSTEM
# (ENSEMBLE + POSITION SIZING + WFV)
# ============================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# ============================================
# LOAD DATA
# ============================================
df = pd.read_csv("panel_dataset_final.csv")

# ============================================
# FEATURE ENGINEERING
# ============================================

df['limit_sentiment'] = df['sent_mean20']
df['limit_trend'] = df['sent_mean20'].diff()

df['continuity_score'] = 1 / (1 + df['sentiment_std'])
df['continuity_local'] = df['avg_sentiment'].diff().abs()

df['discontinuity'] = df['sentiment_shock']

df['momentum'] = df['sentiment_momentum']
df['volatility'] = df['sent_std20']
df['news_impact'] = df['abnormal_news']

# TARGETS
df['target_t1'] = (df['return_t+1'] > 0).astype(int)
df['target_t3'] = (df['return_t+3'] > 0).astype(int)
df['target_t5'] = (df['return_t+5'] > 0).astype(int)

df['ret_t1'] = df['return_t+1']
df['ret_t3'] = df['return_t+3']
df['ret_t5'] = df['return_t+5']

features = [
    'limit_sentiment','limit_trend','continuity_score',
    'continuity_local','discontinuity','momentum',
    'volatility','news_impact','avg_sentiment'
]

df_model = df[features + [
    'target_t1','target_t3','target_t5',
    'ret_t1','ret_t3','ret_t5'
]].dropna().reset_index(drop=True)

# ============================================
# WALK-FORWARD SETTINGS
# ============================================

train_window = 500
test_window = 100
step = 100

all_results = []

# ============================================
# WALK-FORWARD LOOP
# ============================================

for start in range(0, len(df_model) - train_window - test_window, step):

    train_df = df_model.iloc[start:start+train_window]
    test_df  = df_model.iloc[start+train_window:start+train_window+test_window].copy()

    X_train = train_df[features]
    X_test  = test_df[features]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # ============================================
    # TRAIN MODELS
    # ============================================

    model_t1 = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight='balanced', random_state=42)
    model_t3 = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight='balanced', random_state=42)
    model_t5 = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight='balanced', random_state=42)

    model_t1.fit(X_train_scaled, train_df['target_t1'])
    model_t3.fit(X_train_scaled, train_df['target_t3'])
    model_t5.fit(X_train_scaled, train_df['target_t5'])

    # ============================================
    # PREDICTIONS
    # ============================================

    p1 = model_t1.predict_proba(X_test_scaled)[:,1]
    p3 = model_t3.predict_proba(X_test_scaled)[:,1]
    p5 = model_t5.predict_proba(X_test_scaled)[:,1]

    test_df['p1'] = p1
    test_df['p3'] = p3
    test_df['p5'] = p5

    # ============================================
    # 🔥 ENSEMBLE SIGNAL (WEIGHTED)
    # ============================================

    # More weight to longer horizon (trend)
    test_df['ensemble_signal'] = (
        0.2 * test_df['p1'] +
        0.3 * test_df['p3'] +
        0.5 * test_df['p5']
    )

    # Ranking
    test_df['rank'] = test_df['ensemble_signal'].rank(pct=True)

    # ============================================
    # 🔥 TRADE FILTER
    # ============================================

    rank_threshold = test_df['rank'].quantile(0.60)
    cont_threshold = test_df['continuity_score'].quantile(0.30)
    vol_threshold  = test_df['volatility'].quantile(0.85)

    test_df['trade'] = (
        (test_df['rank'] > rank_threshold) &
        (test_df['continuity_score'] > cont_threshold) &
        (test_df['volatility'] < vol_threshold)
    )

    # ============================================
    # 🔥 POSITION SIZING (KEY UPGRADE)
    # ============================================

    # Confidence-based sizing
    test_df['confidence'] = test_df['ensemble_signal']

    # Normalize confidence (0–1)
    test_df['position_size'] = (
        (test_df['confidence'] - test_df['confidence'].min()) /
        (test_df['confidence'].max() - test_df['confidence'].min() + 1e-9)
    )

    # Risk control: cap position size
    test_df['position_size'] = test_df['position_size'].clip(0, 1)

    # Apply only when trade = 1
    test_df['position_size'] = test_df['position_size'] * test_df['trade']

    # ============================================
    # STRATEGY RETURNS (USING POSITION SIZE)
    # ============================================

    # Using T+5 as main return driver
    test_df['strategy'] = test_df['position_size'] * test_df['ret_t5']

    returns = test_df['strategy']

    if returns.std() == 0:
        continue

    # ============================================
    # PERFORMANCE METRICS
    # ============================================

    cumulative_return = (1 + returns).prod()
    sharpe = returns.mean() / returns.std()

    cum_curve = (1 + returns).cumprod()
    drawdown = (cum_curve / cum_curve.cummax()) - 1
    max_dd = drawdown.min()

    all_results.append({
        'start': start,
        'return': cumulative_return,
        'sharpe': sharpe,
        'max_drawdown': max_dd
    })

# ============================================
# FINAL RESULTS
# ============================================

results_df = pd.DataFrame(all_results)

print("\n📊 WALK-FORWARD RESULTS:")
print(results_df)

print("\n🔥 FINAL PERFORMANCE:")
print("Mean Return:", results_df['return'].mean())
print("Mean Sharpe:", results_df['sharpe'].mean())
print("Worst Drawdown:", results_df['max_drawdown'].min())

results_df['cum_return'] = results_df['return'].cumprod()
print("Final Cumulative Return:", results_df['cum_return'].iloc[-1])


📊 WALK-FORWARD RESULTS:
   start    return    sharpe  max_drawdown
0      0  1.006888  0.011607     -0.116608
1    100  1.372110  0.200969     -0.111279
2    200  1.141290  0.091408     -0.129887
3    300  1.248145  0.248065     -0.046279
4    400  0.875247 -0.060221     -0.291252
5    500  1.406087  0.184990     -0.120137
6    600  1.036668  0.034432     -0.170665
7    700  1.559261  0.310732     -0.034538
8    800  1.879618  0.230923     -0.055121
9    900  0.992829  0.001775     -0.156005

🔥 FINAL PERFORMANCE:
Mean Return: 1.251814038931694
Mean Sharpe: 0.12546803873189266
Worst Drawdown: -0.2912517112282428
Final Cumulative Return: 7.305931609375773


In [3]:
# ============================================
# 📌 FINAL ELITE QUANT SYSTEM (ALL FEATURES)
# ============================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# ============================================
# LOAD DATA
# ============================================
df = pd.read_csv("panel_dataset_final.csv")

# ============================================
# FEATURE ENGINEERING
# ============================================

df['limit_sentiment'] = df['sent_mean20']
df['limit_trend'] = df['sent_mean20'].diff()

df['continuity_score'] = 1 / (1 + df['sentiment_std'])
df['continuity_local'] = df['avg_sentiment'].diff().abs()

df['discontinuity'] = df['sentiment_shock']

df['momentum'] = df['sentiment_momentum']
df['volatility'] = df['sent_std20']
df['news_impact'] = df['abnormal_news']

# TARGETS
df['target_t1'] = (df['return_t+1'] > 0).astype(int)
df['target_t3'] = (df['return_t+3'] > 0).astype(int)
df['target_t5'] = (df['return_t+5'] > 0).astype(int)

df['ret_t1'] = df['return_t+1']
df['ret_t3'] = df['return_t+3']
df['ret_t5'] = df['return_t+5']

features = [
    'limit_sentiment','limit_trend','continuity_score',
    'continuity_local','discontinuity','momentum',
    'volatility','news_impact','avg_sentiment'
]

df_model = df[features + [
    'target_t1','target_t3','target_t5',
    'ret_t1','ret_t3','ret_t5'
]].dropna().reset_index(drop=True)

# ============================================
# WALK-FORWARD SETTINGS
# ============================================

train_window = 500
test_window = 100
step = 100

all_results = []

# ============================================
# WALK-FORWARD LOOP
# ============================================

for start in range(0, len(df_model) - train_window - test_window, step):

    train_df = df_model.iloc[start:start+train_window]
    test_df  = df_model.iloc[start+train_window:start+train_window+test_window].copy()

    X_train = train_df[features]
    X_test  = test_df[features]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # ============================================
    # TRAIN MODELS
    # ============================================

    model_t1 = RandomForestClassifier(n_estimators=400, max_depth=8, class_weight='balanced', random_state=42)
    model_t3 = RandomForestClassifier(n_estimators=400, max_depth=8, class_weight='balanced', random_state=42)
    model_t5 = RandomForestClassifier(n_estimators=400, max_depth=8, class_weight='balanced', random_state=42)

    model_t1.fit(X_train_scaled, train_df['target_t1'])
    model_t3.fit(X_train_scaled, train_df['target_t3'])
    model_t5.fit(X_train_scaled, train_df['target_t5'])

    # ============================================
    # PREDICTIONS
    # ============================================

    p1 = model_t1.predict_proba(X_test_scaled)[:,1]
    p3 = model_t3.predict_proba(X_test_scaled)[:,1]
    p5 = model_t5.predict_proba(X_test_scaled)[:,1]

    test_df['p1'] = p1
    test_df['p3'] = p3
    test_df['p5'] = p5

    # ============================================
    # 🔥 ENSEMBLE SIGNAL (TREND FOCUSED)
    # ============================================

    test_df['ensemble_signal'] = (
        0.1 * test_df['p1'] +
        0.3 * test_df['p3'] +
        0.6 * test_df['p5']
    )

    # 🔥 Smooth signal (Sharpe improvement)
    test_df['ensemble_signal'] = test_df['ensemble_signal'].rolling(3).mean()

    # Ranking
    test_df['rank'] = test_df['ensemble_signal'].rank(pct=True)

    # ============================================
    # 🔥 TRADE FILTER (WITH REGIME CONTROL)
    # ============================================

    rank_threshold = test_df['rank'].quantile(0.60)
    cont_threshold = test_df['continuity_score'].quantile(0.30)
    vol_threshold  = test_df['volatility'].quantile(0.75)

    test_df['trade'] = (
        (test_df['rank'] > rank_threshold) &
        (test_df['continuity_score'] > cont_threshold) &
        (test_df['volatility'] < vol_threshold)
    )

    # ============================================
    # 🔥 POSITION SIZING (NON-LINEAR)
    # ============================================

    test_df['confidence'] = test_df['ensemble_signal']

    test_df['position_size'] = (
        (test_df['confidence'] - test_df['confidence'].min()) /
        (test_df['confidence'].max() - test_df['confidence'].min() + 1e-9)
    )

    test_df['position_size'] = (test_df['position_size'] ** 2)  # nonlinear boost
    test_df['position_size'] = test_df['position_size'].clip(0, 1)
    test_df['position_size'] = test_df['position_size'] * test_df['trade']

    # ============================================
    # STRATEGY
    # ============================================

    test_df['strategy'] = test_df['position_size'] * test_df['ret_t5']

    # ============================================
    # 🔥 DRAWDOWN CONTROL
    # ============================================

    cum_curve = (1 + test_df['strategy']).cumprod()
    drawdown = (cum_curve / cum_curve.cummax()) - 1

    test_df['strategy'] = np.where(drawdown < -0.2, 0, test_df['strategy'])

    returns = test_df['strategy']

    if returns.std() == 0:
        continue

    # ============================================
    # PERFORMANCE
    # ============================================

    cumulative_return = (1 + returns).prod()
    sharpe = returns.mean() / returns.std()

    cum_curve = (1 + returns).cumprod()
    drawdown = (cum_curve / cum_curve.cummax()) - 1
    max_dd = drawdown.min()

    all_results.append({
        'start': start,
        'return': cumulative_return,
        'sharpe': sharpe,
        'max_drawdown': max_dd
    })

# ============================================
# FINAL RESULTS
# ============================================

results_df = pd.DataFrame(all_results)

print("\n📊 WALK-FORWARD RESULTS:")
print(results_df)

print("\n🔥 FINAL PERFORMANCE:")
print("Mean Return:", results_df['return'].mean())
print("Mean Sharpe:", results_df['sharpe'].mean())
print("Worst Drawdown:", results_df['max_drawdown'].min())

results_df['cum_return'] = results_df['return'].cumprod()
print("Final Cumulative Return:", results_df['cum_return'].iloc[-1])


📊 WALK-FORWARD RESULTS:
   start    return    sharpe  max_drawdown
0      0  1.104377  0.134750     -0.049036
1    100  1.185290  0.185346     -0.046972
2    200  0.973715 -0.031419     -0.070860
3    300  1.078052  0.168851     -0.030407
4    400  0.973406 -0.013254     -0.190334
5    500  1.161426  0.170724     -0.046055
6    600  0.981641 -0.017776     -0.127790
7    700  1.568780  0.340449     -0.020350
8    800  1.616482  0.226313     -0.052452
9    900  0.916337 -0.095207     -0.143534

🔥 FINAL PERFORMANCE:
Mean Return: 1.1559504858849687
Mean Sharpe: 0.10687766782196777
Worst Drawdown: -0.1903343763695704
Final Cumulative Return: 3.5435575419563854


In [5]:
# ============================================
# 📌 ULTRA-OPTIMIZED SHARPE-FOCUSED SYSTEM
# ============================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# ============================================
# LOAD DATA
# ============================================
df = pd.read_csv("panel_dataset_final.csv")

# ============================================
# FEATURE ENGINEERING
# ============================================

df['limit_sentiment'] = df['sent_mean20']
df['limit_trend'] = df['sent_mean20'].diff()

df['continuity_score'] = 1 / (1 + df['sentiment_std'])
df['continuity_local'] = df['avg_sentiment'].diff().abs()

df['discontinuity'] = df['sentiment_shock']

df['momentum'] = df['sentiment_momentum']
df['volatility'] = df['sent_std20']
df['news_impact'] = df['abnormal_news']

# TARGETS
df['target_t1'] = (df['return_t+1'] > 0).astype(int)
df['target_t3'] = (df['return_t+3'] > 0).astype(int)
df['target_t5'] = (df['return_t+5'] > 0).astype(int)

df['ret_t5'] = df['return_t+5']

features = [
    'limit_sentiment','limit_trend','continuity_score',
    'continuity_local','discontinuity','momentum',
    'volatility','news_impact','avg_sentiment'
]

df_model = df[features + [
    'target_t1','target_t3','target_t5','ret_t5'
]].dropna().reset_index(drop=True)

# ============================================
# WALK-FORWARD SETTINGS
# ============================================

train_window = 500
test_window = 100
step = 100

all_results = []

# ============================================
# WALK-FORWARD LOOP
# ============================================

for start in range(0, len(df_model) - train_window - test_window, step):

    train_df = df_model.iloc[start:start+train_window]
    test_df  = df_model.iloc[start+train_window:start+train_window+test_window].copy()

    X_train = train_df[features]
    X_test  = test_df[features]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # ============================================
    # TRAIN MODELS (MULTI-HORIZON)
    # ============================================

    model_t1 = RandomForestClassifier(n_estimators=400, max_depth=8, class_weight='balanced', random_state=42)
    model_t3 = RandomForestClassifier(n_estimators=400, max_depth=8, class_weight='balanced', random_state=42)
    model_t5 = RandomForestClassifier(n_estimators=400, max_depth=8, class_weight='balanced', random_state=42)

    model_t1.fit(X_train_scaled, train_df['target_t1'])
    model_t3.fit(X_train_scaled, train_df['target_t3'])
    model_t5.fit(X_train_scaled, train_df['target_t5'])

    # ============================================
    # PREDICTIONS
    # ============================================

    p1 = model_t1.predict_proba(X_test_scaled)[:,1]
    p3 = model_t3.predict_proba(X_test_scaled)[:,1]
    p5 = model_t5.predict_proba(X_test_scaled)[:,1]

    # ============================================
    # 🔥 ENSEMBLE (TREND DOMINANT)
    # ============================================

    test_df['signal'] = (
        0.1 * p1 +
        0.3 * p3 +
        0.6 * p5
    )

    # 🔥 STRONG SMOOTHING (KEY FOR SHARPE)
    test_df['signal'] = test_df['signal'].rolling(5).mean()

    # Ranking
    test_df['rank'] = test_df['signal'].rank(pct=True)

    # ============================================
    # 🔥 STRICT TRADE FILTER (HIGH QUALITY ONLY)
    # ============================================

    rank_threshold = test_df['rank'].quantile(0.70)   # top 30%
    cont_threshold = test_df['continuity_score'].quantile(0.40)
    vol_threshold  = test_df['volatility'].quantile(0.70)

    test_df['confidence'] = test_df['signal']

    test_df['trade'] = (
        (test_df['rank'] > rank_threshold) &
        (test_df['continuity_score'] > cont_threshold) &
        (test_df['volatility'] < vol_threshold) &
        (test_df['confidence'] > 0.60)
    )

    # ============================================
    # 🔥 POSITION SIZING (AGGRESSIVE FILTERED)
    # ============================================

    norm_conf = (
        (test_df['confidence'] - test_df['confidence'].min()) /
        (test_df['confidence'].max() - test_df['confidence'].min() + 1e-9)
    )

    test_df['position_size'] = np.where(
        test_df['trade'],
        norm_conf ** 2,
        0
    )

    # ============================================
    # STRATEGY
    # ============================================

    test_df['strategy'] = test_df['position_size'] * test_df['ret_t5']

    # ============================================
    # 🔥 DRAWDOWN STOP (STRICT)
    # ============================================

    cum_curve = (1 + test_df['strategy']).cumprod()
    drawdown = (cum_curve / cum_curve.cummax()) - 1

    test_df['strategy'] = np.where(drawdown < -0.15, 0, test_df['strategy'])

    returns = test_df['strategy']

    if returns.std() == 0:
        continue

    # ============================================
    # PERFORMANCE
    # ============================================

    cumulative_return = (1 + returns).prod()
    sharpe = returns.mean() / returns.std()

    cum_curve = (1 + returns).cumprod()
    drawdown = (cum_curve / cum_curve.cummax()) - 1
    max_dd = drawdown.min()

    all_results.append({
        'start': start,
        'return': cumulative_return,
        'sharpe': sharpe,
        'max_drawdown': max_dd
    })

# ============================================
# FINAL RESULTS
# ============================================

results_df = pd.DataFrame(all_results)

print("\n📊 WALK-FORWARD RESULTS:")
print(results_df)

print("\n🔥 FINAL SHARPE-FOCUSED PERFORMANCE:")
print("Mean Return:", results_df['return'].mean())
print("Mean Sharpe:", results_df['sharpe'].mean())
print("Worst Drawdown:", results_df['max_drawdown'].min())

results_df['cum_return'] = results_df['return'].cumprod()
print("Final Cumulative Return:", results_df['cum_return'].iloc[-1])


📊 WALK-FORWARD RESULTS:
   start    return    sharpe  max_drawdown
0      0  0.982804 -0.097284     -0.023048
1    200  1.054156  0.083574     -0.030103
2    400  0.904706 -0.138405     -0.095294
3    500  1.006296  0.100000      0.000000
4    600  1.097588  0.171627     -0.009218
5    700  1.420737  0.295638     -0.006697
6    800  1.576333  0.153967     -0.035227
7    900  0.945044 -0.134340     -0.061096

🔥 FINAL SHARPE-FOCUSED PERFORMANCE:
Mean Return: 1.1234579510410323
Mean Sharpe: 0.05434707900880099
Worst Drawdown: -0.09529443754540523
Final Cumulative Return: 2.1910779299289147
